# Chapter 13. 로봇공학 기초와 Sim-to-Real — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter13_1_sim_to_real.ipynb)

책 본문: [13.1 로봇공학 기초와 Sim-to-Real](https://smhanlab.com/book-ml/kor/ml2/chapter13/1.html)

이 노트북은 13.1절의 핵심 두 가지를 숫자로 확인합니다:

1. **2관절 로봇팔의 정규구학/역기구학** — `l1 = l2 = 0.5` m 팔의
   팔 끝 좌표 계산, 그리고 같은 목표에 대한 *두 가지* 해
   (elbow-up / elbow-down).
2. **현실 격차(reality gap)와 도메인 무작위화** — 진자 동역학에
   "감쇠(마찰) 계수 `mu`"라는 한 개만 다른 파라미터를 넣고,
   20개의 PD 정책의 점수가 `mu`가 바뀜에 따라 어떻게
   달라지는지 그리드로 측정합니다.

numpy/matplotlib/gymnasium만 씁니다 — torch 불필요, 외부
데이터 다운로드 불필요.


## 1. 환경 준비

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import gymnasium as gym

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("gymnasium", gym.__version__, "| 그림 저장 위치:", IMG)


gymnasium 1.3.0 | 그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 2. 2관절 로봇팔: 정규구학(FK)과 역기구학(IK)

마디 길이 `l1`, `l2`, 첫 마디 각도 `theta1`, 두 번째 마디의
*상대* 각도 `theta2`이면, 팔 끝 좌표는 두 마디의 기여를 더한 것:

    x = l1*cos(theta1) + l2*cos(theta1+theta2)
    y = l1*sin(theta1) + l2*sin(theta1+theta2)


In [2]:
def fk(t1, t2, l1=0.5, l2=0.5):
    return (l1*np.cos(t1) + l2*np.cos(t1+t2),
            l1*np.sin(t1) + l2*np.sin(t1+t2))

def fk_elbow(t1, l1=0.5):
    return (l1*np.cos(t1), l1*np.sin(t1))

def ik(x, y, l1=0.5, l2=0.5):
    """두 해를 반환: (t1_up, t2_up, t1_down, t2_down)"""
    d = np.hypot(x, y)
    c2 = np.clip((d*d - l1*l1 - l2*l2) / (2*l1*l2), -1.0, 1.0)
    s2 = np.sqrt(max(0.0, 1.0 - c2*c2))
    a  = np.arctan2(y, x)
    t1_up,   t2_up   = a - np.arctan2(l2*s2, l1 + l2*c2),  np.arctan2(s2, c2)
    t1_down, t2_down = a + np.arctan2(l2*s2, l1 + l2*c2), -np.arctan2(s2, c2)
    return t1_up, t2_up, t1_down, t2_down

# 본문 예: l1=l2=0.5, theta1=30도, theta2=60도
t1, t2 = np.deg2rad(30), np.deg2rad(60)
elbow = fk_elbow(t1)
tip   = fk(t1, t2)
print("elbow =", np.round(elbow, 3), " tip =", np.round(tip, 3))

# 본문 예: 목표 (0.5, 0.5)의 두 해
t1u, t2u, t1d, t2d = ik(0.5, 0.5)
print("elbow-up  : (t1, t2) = (%.1f도, %+.1f도)" % (np.degrees(t1u), np.degrees(t2u)))
print("elbow-down: (t1, t2) = (%.1f도, %+.1f도)" % (np.degrees(t1d), np.degrees(t2d)))
print("검증 FK(up)   =", np.round(fk(t1u, t2u), 6))
print("검증 FK(down) =", np.round(fk(t1d, t2d), 6))
a_u, b_u, a_d, b_d = ik(1.0, 0.0)
print("가장 먼 점 (1.0, 0.0): 두 해가 중첩 -> elbow-up (t1,t2) = (%.2f도, %+.2f도), elbow-down (%.2f도, %+.2f도)"
      % (np.degrees(a_u), np.degrees(b_u), np.degrees(a_d), np.degrees(b_d)))


elbow = [0.433 0.25 ]  tip = [0.433 0.75 ]
elbow-up  : (t1, t2) = (0.0도, +90.0도)
elbow-down: (t1, t2) = (90.0도, -90.0도)
검증 FK(up)   = [0.5 0.5]
검증 FK(down) = [0.5 0.5]
가장 먼 점 (1.0, 0.0): 두 해가 중첩 -> elbow-up (t1,t2) = (0.00도, +0.00도), elbow-down (0.00도, -0.00도)


**그림 1** — 왼쪽: 도달 가능 영역(반경 `l1+l2 = 1` 원판)과,
`theta1 = 30도` 고정에서 `theta2`를 0~120도로 돌렸을 때 팔 끝
궤적. 오른쪽: 목표 `(0.5, 0.5)`에 대한 IK의 두 해 —
elbow-up(파랑)과 elbow-down(주황)이 *같은* 팔 끝 위치로
도달한다.


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ax = axes[0]
th = np.linspace(0, 2*np.pi, 200)
ax.fill(np.cos(th), np.sin(th), color="#eef3fb")
ax.plot(np.cos(th), np.sin(th), "k-", lw=1)
ax.set_title("FK: 도달 가능 영역 (l1=l2=0.5)")
t1 = np.deg2rad(30)
for t2deg, ls in ((60, "-"),):
    pass
t2s = np.linspace(0, np.deg2rad(120), 60)
xt, yt = zip(*[fk(t1, t2) for t2 in t2s])
ax.plot(xt, yt, "-", lw=2, color="#2c6fbb")
ax.annotate("tip $\\theta_2$를 돌릴 때\n팔 끝 궤적", (0.433, 0.75),
            xytext=(0.55, 0.85), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="gray"))
ex, ey = fk_elbow(t1); tx, ty = fk(t1, np.deg2rad(60))
ax.plot([0, ex, tx], [0, ey, ty], "o-", color="#2c6fbb", lw=2)
ax.plot([0], [0], "ks", ms=6); ax.plot([ex], [ey], "ko", ms=5)
ax.annotate("elbow (0.866, 0.500)", (ex, ey), xytext=(0.75, 0.30),
            fontsize=8, arrowprops=dict(arrowstyle="->", color="gray"))
ax.annotate("tip (0.433, 0.750)", (tx, ty), xytext=(-0.15, 0.90),
            fontsize=8, arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_aspect("equal"); ax.grid(alpha=0.3)

ax = axes[1]
for (t1a, t2a, c, lab) in ((ik(0.5, 0.5)[0], ik(0.5, 0.5)[1], "#1f77b4", "elbow-up: (0\u00b0, +90\u00b0)"),
                           (ik(0.5, 0.5)[2], ik(0.5, 0.5)[3], "#ff7f0e", "elbow-down: (90\u00b0, -90\u00b0)")):
    ex, ey = fk_elbow(t1a); tx, ty = fk(t1a, t2a)
    ax.plot([0, ex, tx], [0, ey, ty], "o-", color=c, lw=2, label=lab)
ax.plot([0.5], [0.5], "r*", ms=14, label="goal (0.5, 0.5)")
ax.plot([0], [0], "ks", ms=6)
ax.legend(fontsize=9, loc="lower left")
ax.set_title("IK: 같은 목표, 두 자세")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_aspect("equal"); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(f"{IMG}/ch13_1_fk_2link_arm.svg", bbox_inches="tight")
plt.show()
print("저장:", f"{IMG}/ch13_1_fk_2link_arm.svg")


저장: /home/smhan/book-ml/kor/src/images/ch13_1_fk_2link_arm.svg


## 3. 진자 동역학: 시뮬레이터의 정체

`Pendulum-v1`(13.2절의 주인공)은 아래 미분방정식을 오일러
적분(`dt=0.05`)으로 스텝합니다 (`g=m=l=1`, 최대 토크 2.0,
최대 각속도 8.0):

$\ddot\theta = 15\sin\theta + 3u$

여기서 `mu`를 곱한 감쇠항 $-\mu\dot\theta$를 *추가*한
버전을 "실물 로봇"으로 삼아, 한 개만 다른 파라미터가 만드는
차이를 봅니다.


In [4]:
g, m, l, dt, MAXS, MAXU = 10.0, 1.0, 1.0, 0.05, 8.0, 2.0

def make_sim(mu):
    """Pendulum-v1 동역학 + 감쇠 mu. (step, reward, reset) 반환."""
    def step(th, thdot, u):
        u = float(np.clip(u, -MAXU, MAXU))
        thdd = (3*g/(2*l))*np.sin(th) + (3.0/(m*l*l))*u - mu*thdot
        thdot = float(np.clip(thdot + thdd*dt, -MAXS, MAXS))
        return th + thdot*dt, thdot
    def reward(th, thdot, u):
        tn = ((th + np.pi) % (2*np.pi)) - np.pi   # 0 = 위쪽 수직
        return -(tn**2 + 0.1*thdot**2 + 0.001*u**2)
    def reset(rng):
        return float(rng.uniform(-np.pi, np.pi)), float(rng.uniform(-1.0, 1.0))
    return step, reward, reset

# --- 검증: mu=0.1 시뮬레이터(아래 실험용)는 Pendulum-v1 원본(mu=0)
# --- 의 어떤 인스턴스도 아니므로, 대신 '동역학 식 자체'를
# --- 원본 env와 50스텝 비교해 재현했음을 확인한다.
env = gym.make("Pendulum-v1")
obs, _ = env.reset(seed=0)
th, thd = float(np.arctan2(obs[1], obs[0])), float(obs[2])
maxdiff = 0.0
for _ in range(50):
    u = float(np.clip(-1.3*th - 0.4*thd, -MAXU, MAXU))
    o, *_ = env.step([u])
    thdd = (3*g/(2*l))*np.sin(th) + (3.0/(m*l*l))*u   # mu=0 (Pendulum-v1 원본)
    thd  = float(np.clip(thd + thdd*dt, -MAXS, MAXS))
    th   = th + thd*dt
    maxdiff = max(maxdiff, abs(o[2]-thd), abs(o[0]-np.cos(th)), abs(o[1]-np.sin(th)))
print("동역학 재현 검증: 50스텝 관측값 vs 직접 계산, 최대 차이 =", maxdiff)
env.close()
sim  = make_sim(0.1)   # "시뮬레이션"
robot = make_sim(5.0)  # "실물 로봇" (마찰이 50배 큰 세계)


동역학 재현 검증: 50스텝 관측값 vs 직접 계산, 최대 차이 = 4.7683716e-07


## 4. 현실 격차 그리드 실험

**20개 PD 정책** ($k_p \in \{1,2,4,8\}$,
$k_d \in \{0, 0.5, 1, 2, 4\}$, 12 시드 평균)을 **6개 환경**
($\mu \in \{0.0, 0.1, 0.5, 1, 2, 5\}$)에서 200스텝
누적 보상으로 평가합니다.


In [5]:
import pandas as pd

kps, kds = [1, 2, 4, 8], [0.0, 0.5, 1.0, 2.0, 4.0]
pols = [(kp, kd) for kp in kps for kd in kds]
mus  = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]
sims = {mu: make_sim(mu) for mu in mus}
SEEDS = range(12)

def episode(sim, kp, kd, seed, steps=200):
    step, reward, reset = sim
    rng = np.random.default_rng(seed)
    th, thd = reset(rng); tot = 0.0
    for _ in range(steps):
        u = float(np.clip(-kp*th - kd*thd, -MAXU, MAXU))
        tot += reward(th, thd, u)
        th, thd = step(th, thd, u)
    return tot

rows = []
for p in pols:
    rows.append({"kp": p[0], "kd": p[1],
                 **{f"mu={mu}": float(np.mean([episode(sims[mu], *p, s) for s in SEEDS])) for mu in mus}})
tab = pd.DataFrame(rows).set_index(["kp", "kd"]).round(2)
print(tab)


         mu=0.0   mu=0.1   mu=0.5   mu=1.0   mu=2.0   mu=5.0
kp kd                                                       
1  0.0 -1266.19 -1355.31 -1449.17 -1458.79 -1458.48 -1422.29
   0.5 -1452.57 -1456.20 -1457.15 -1455.63 -1451.91 -1412.66
   1.0 -1465.21 -1462.99 -1456.47 -1453.96 -1447.69 -1405.72
   2.0 -1455.55 -1454.72 -1452.88 -1451.02 -1441.71 -1394.46
   4.0 -1447.70 -1448.83 -1449.55 -1446.62 -1432.52 -1376.77
2  0.0 -1224.21 -1345.07 -1444.63 -1452.26 -1449.61 -1407.79
   0.5 -1312.01 -1357.94 -1441.51 -1448.21 -1444.51 -1400.76
   1.0 -1380.22 -1403.19 -1440.50 -1444.07 -1439.67 -1394.34
   2.0 -1453.92 -1450.84 -1440.66 -1436.28 -1429.28 -1381.57
   4.0 -1434.57 -1432.27 -1427.60 -1423.64 -1411.87 -1360.07
4  0.0 -1195.13 -1337.41 -1429.39 -1435.10 -1428.98 -1367.85
   0.5 -1213.06 -1334.64 -1424.55 -1428.31 -1418.91 -1352.62
   1.0 -1219.29 -1327.55 -1415.52 -1417.64 -1406.25 -1336.55
   2.0 -1305.75 -1337.71 -1389.79 -1390.75 -1377.18 -1303.21
   4.0 -1354.91 -1361.53

무작위 정책 비교 기준(본문 `Pendulum-v1` 환경, 12 시드):


In [6]:
env = gym.make("Pendulum-v1")
refs = []
for s in range(12):
    o, _ = env.reset(seed=9000+s); tot = 0.0
    for _ in range(200):
        o, r, term, trunc, _ = env.step(env.action_space.sample()); tot += r
    refs.append(tot)
print("무작위 정책 200스텝 평균 (12 시드):", round(float(np.mean(refs)), 2))
env.close()


무작위 정책 200스텝 평균 (12 시드): -1184.22


In [7]:
# 본문 표와 동일한 요약
simcol, robcol = "mu=0.1", "mu=5.0"
sim_scores = tab[simcol]                 # MultiIndex (kp,kd) 유지
rob_scores = tab[robcol]
best_sim = sim_scores.idxmax()
best_rob = rob_scores.idxmax()
rank_sim = sim_scores.rank(ascending=False)
rank_rob = rob_scores.rank(ascending=False)
print(f"시뮬레이션(mu=0.1) 최고: kp={best_sim[0]}, kd={best_sim[1]:g}  -> {sim_scores.loc[best_sim]:.1f}")
print(f"실물(mu=5.0)        최고: kp={best_rob[0]}, kd={best_rob[1]:g}  -> {rob_scores.loc[best_rob]:.1f}")
print(f"점수 범위: 시뮬레이션 [{sim_scores.min():.1f}, {sim_scores.max():.1f}] | 실물 [{rob_scores.min():.1f}, {rob_scores.max():.1f}]")
print(f"점수 평균 차이 (실물 - 시뮬레이션): {rob_scores.mean() - sim_scores.mean():+.2f}")
ra = rank_sim.to_numpy(); rb = rank_rob.to_numpy()
print("점수 순위의 상관계수 (20개 정책):", round(float(np.corrcoef(ra, rb)[0,1]), 3))
p = (1, 0.0)
print(f"정책 {p}: 시뮬레이션 {int(rank_sim.loc[p])}위 -> 실물 {int(rank_rob.loc[p])}위  "
      f"({sim_scores.loc[p]:.1f} -> {rob_scores.loc[p]:.1f})")
# 시뮬레이션 최고 정책 (8, 2)의 mu별 최악
row82 = tab.loc[(8, 2.0)]
wc = row82[[f"mu={mu}" for mu in mus[1:]]].idxmin()
print("시뮬레이션 최고 정책 (8, 2)의 mu별 점수: " +
      "  ".join(f"mu={mu}: {row82[f'mu={mu}']:.1f}" for mu in mus[1:]))
print("  -> 최악은", wc, "에서")


시뮬레이션(mu=0.1) 최고: kp=8, kd=2  -> -1163.9
실물(mu=5.0)        최고: kp=8, kd=0  -> -1204.3
점수 범위: 시뮬레이션 [-1463.0, -1163.9] | 실물 [-1422.3, -1204.3]
점수 평균 차이 (실물 - 시뮬레이션): +5.02
점수 순위의 상관계수 (20개 정책): 0.759
정책 (1, 0.0): 시뮬레이션 11위 -> 실물 20위  (-1355.3 -> -1422.3)
시뮬레이션 최고 정책 (8, 2)의 mu별 점수: mu=0.1: -1163.9  mu=0.5: -1222.2  mu=1.0: -1225.9  mu=2.0: -1225.1  mu=5.0: -1204.3
  -> 최악은 mu=1.0 에서


**그림 2** — 왼쪽: 20개 정책의 점수(200스텝 평균)가 `mu`에 따라
어떻게 떨어지는지(한 점선 = 하나의 PD 정책, 색 = $k_p$).
오른쪽: 시뮬레이션($\mu=0.1$) 점수 vs 실물($\mu=5.0$) 점수 —
대각선 $y=x$에서 벗어난 만큼이 바로 현실 격차.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
colors = {1: "#d62728", 2: "#ff7f0e", 4: "#2ca02c", 8: "#1f77b4"}
ax = axes[0]
for (kp, kd) in pols:
    ax.plot(mus[1:], tab.loc[(kp, kd), [f"mu={mu}" for mu in mus[1:]]].to_numpy(),
            "-", color=colors[kp], alpha=0.85, lw=1.8,
            label=f"kp={kp}, kd={kd:g}")
ax.set_xscale("log"); ax.set_xticks(mus[1:])
ax.set_xticklabels([str(m) for m in mus[1:]])
ax.set_xlabel("감쇠(마찰) 계수 $\\mu$"); ax.set_ylabel("200스텝 평균 보상")
ax.set_title("점수가 mu에 따라 어떻게 달라지는가")
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc="lower left")

ax = axes[1]
ax.scatter(sim_scores, rob_scores, s=45, color="#1f77b4", zorder=3,
           edgecolors="k", linewidths=0.4)
for (kp, kd) in pols:
    ax.annotate(f"({kp},{kd:g})", tab.loc[(kp, kd), [simcol, robcol]].to_numpy(),
                textcoords="offset points", xytext=(4, 3), fontsize=6.5, color="0.3")
lim = [min(sim_scores.min(), rob_scores.min()) - 30,
       max(sim_scores.max(), rob_scores.max()) + 30]
ax.plot(lim, lim, "k--", lw=1, zorder=2, label="$y=x$ (격차 0)")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("시뮬레이션 ($\\mu=0.1$) 점수")
ax.set_ylabel("실물 ($\\mu=5.0$) 점수")
ax.set_title("시뮬레이션 점수 vs 실물 점수")
ax.legend(fontsize=9, loc="upper right"); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(f"{IMG}/ch13_1_domain_randomization.svg", bbox_inches="tight")
plt.show()
print("저장:", f"{IMG}/ch13_1_domain_randomization.svg")


저장: /home/smhan/book-ml/kor/src/images/ch13_1_domain_randomization.svg


현실 격차가 *고정 오프셋*이 아니라 정책마다 다르다는 것을직접 확인:

In [9]:
off = (tab[robcol] - tab[simcol]).sort_values(ascending=False)
print("정책별 점수 차 (실물 - 시뮬레이션), 큰 것부터:")
print(off.round(1))

정책별 점수 차 (실물 - 시뮬레이션), 큰 것부터:
kp  kd 
4   4.0    103.9
2   4.0     72.2
1   4.0     72.1
2   2.0     69.3
1   2.0     60.3
    1.0     57.3
    0.5     43.5
4   2.0     34.5
2   1.0      8.9
4   1.0     -9.0
    0.5    -18.0
    0.0    -30.4
8   4.0    -31.8
    0.0    -38.7
    0.5    -40.3
    1.0    -40.3
    2.0    -40.4
2   0.5    -42.8
    0.0    -62.7
1   0.0    -67.0
dtype: float64


## 5. 정리

- **IK 해가 여러 개** — 같은 목표 위치(0.5, 0.5)를 elbow-up
  (0도, +90도)와 elbow-down (90도, -90도)로 만들 수 있음.
  정책이 상태 *전체*를 봐야 하는 이유.
- **시뮬레이션 = 미분방정식을 dt로 끊어 반복** — `mu=0`이면
  `Pendulum-v1`과 동일(검증: 50스텝 최대 차이 1e-5 이하).
- **현실 격차는 고정 오프셋이 아니라 정책별 편차** —
  `mu`만 0.1 -> 5.0으로 바뀌어도, 정책별 점수 차(실물-시뮬)는
  -67.0 ~ +103.9 포인트까지 벌어지고(평균 +5.02), 정책 순위
  도 바뀌음((1, 0)은 시뮬레이션 11위 -> 실물 20위). 데이터로
  줄이지 못한다.
- **도메인 무작위화** = `mu`를 범위 전체에서 매 에피소드마다
  뽑아 학습 → "어느 mu에서도 최악이 아닌" 정책. 시뮬레이션
  최고 정책 (8, 2)의 최악 환경은 우리가 가정한
  실물(5.0)도 아닌 **mu=1.0**(−1225.9)이라는 점을 기억할 것.

다음 절(13.2)에서는 이 시뮬레이션 위에서 Gymnasium의 로봇
환경을 직접 탐색합니다.
